In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")

In [2]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

In [3]:
# sort theo timestamp và bỏ đi các cột không đóng góp vào phân loại
train_df = train_df.sort_values(by=['timestamp_num'])
valid_df = valid_df.sort_values(by=['timestamp_num'])
test_df = test_df.sort_values(by=['timestamp_num'])

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [4]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import copy
import warnings
warnings.filterwarnings('ignore')

# tách label ra làm y (target) và phần còn lại là X (features)
target_col = 'label'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# số class trong tập train
n_classes = len(np.unique(y_train))

print(f"Số lượng lớp (classes) phát hiện được: {n_classes}")
print(f"Các nhãn (labels): {np.unique(y_train)}")

Số lượng lớp (classes) phát hiện được: 13
Các nhãn (labels): [ 0  1  2  3  4  5  6  7  8  9 10 11 12]


In [5]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import numpy as np

# Khởi tạo lại các Base Learners với XGBoost, CatBoost dùng GPU.
# LightGBM chuyển về CPU để tránh lỗi chia nhánh (left_count > 0) trên GPU.
xgb_model_gpu = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, tree_method='hist', device='cuda', random_state=42)
cat_model_gpu = cb.CatBoostClassifier(iterations=100, learning_rate=0.05, random_state=42, verbose=False, task_type='GPU')
lgb_model_cpu = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42, n_jobs=-1) 
ada_model_cpu = AdaBoostClassifier(n_estimators=100, random_state=42) # AdaBoost mặc định chạy CPU

base_models_gpu = {
    'XGBoost': xgb_model_gpu,
    'CatBoost': cat_model_gpu,
    'LightGBM': lgb_model_cpu,
    'AdaBoost': ada_model_cpu
}

meta_model_rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

n_classes = len(np.unique(y_train))

print("--- HUẤN LUYỆN STACKING THEO PHƯƠNG PHÁP TRAIN-VALID-TEST (HYBRID GPU/CPU) ---")

# 1. Khởi tạo ma trận rỗng để chứa đặc trưng Meta cho Valid và Test
meta_X_valid = np.zeros((len(X_valid), len(base_models_gpu) * n_classes))
meta_X_test = np.zeros((len(X_test), len(base_models_gpu) * n_classes))

print("\n--- BƯỚC 1: HUẤN LUYỆN BASE MODELS TRÊN TẬP TRAIN ---")
for name, model in base_models_gpu.items():
    print(f"> Đang huấn luyện {name}...")
    col_start = list(base_models_gpu.keys()).index(name) * n_classes
    col_end = col_start + n_classes
    
    # Fit các base model trên tập Train (Dùng thêm Valid để làm early stopping nếu hỗ trợ)
    if name == 'XGBoost':
        model.set_params(early_stopping_rounds=30)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    elif name == 'CatBoost':
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], early_stopping_rounds=30, verbose=False)
    elif name == 'LightGBM':
        callbacks = [lgb.early_stopping(stopping_rounds=30, verbose=False)]
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=callbacks)
    else:
        model.fit(X_train, y_train)
        
    # Tạo dự đoán xác suất (predict_proba) trên Valid và Test để làm Meta-features
    meta_X_valid[:, col_start:col_end] = model.predict_proba(X_valid)
    meta_X_test[:, col_start:col_end] = model.predict_proba(X_test)

print("\n--- BƯỚC 2: HUẤN LUYỆN META-LEARNER ---")
# Trong cách đơn giản này, ta lấy dự đoán trên tập Valid (được tạo ra bởi models đã học trên Train)
# để làm đầu vào (features) huấn luyện Meta-Learner
meta_model_rf.fit(meta_X_valid, y_valid)

from sklearn.metrics import roc_auc_score

print("\n--- BƯỚC 3: ĐÁNH GIÁ TRÊN TẬP TEST ---")
# Sử dụng meta_X_test để đưa ra dự đoán cuối cùng
y_pred_test = meta_model_rf.predict(meta_X_test)
y_pred_proba = meta_model_rf.predict_proba(meta_X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")

if n_classes == 2:
    auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])
else:
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
print(f"AUC-ROC:  {auc_roc:.4f}")

print(classification_report(y_test, y_pred_test, digits=4))

--- HUẤN LUYỆN STACKING THEO PHƯƠNG PHÁP TRAIN-VALID-TEST (HYBRID GPU/CPU) ---

--- BƯỚC 1: HUẤN LUYỆN BASE MODELS TRÊN TẬP TRAIN ---
> Đang huấn luyện XGBoost...
> Đang huấn luyện CatBoost...
> Đang huấn luyện LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.439213 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41835
[LightGBM] [Info] Number of data points in the train set: 3294945, number of used features: 314
[LightGBM] [Info] Start training from score -2.594813
[LightGBM] [Info] Start training from score -1.681426
[LightGBM] [Info] Start training from score -3.576662
[LightGBM] [Info] Start training from score -1.785067
[LightGBM] [Info] Start training from score -2.481586
[LightGBM] [Info] Start training from score -2.644824
[LightGBM] [Info] Start training from score -1.564204
[LightGBM] [Info] Start training from scor